# Group Information

**Git repository:** https://github.com/MagicBOTAlex/Social-Informatik-02467-project

**Group member responsibility:**
- Laibah Manoor Zia Choudhary (s244320)
- Kaitlyn Wu Brooks (s246082)
- Zhentao Wei (s246213)

We sat in a Discord call for multiple hours and worked hard, twice

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations
import random
import math
from collections import defaultdict, Counter
import ast
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
from vader_sentiment.vader_sentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
import os
import ast
import logging
import pycountry
from multiprocessing import Pool
import json
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True, nb_workers=16)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


## Ensure dataset in ./data

In [2]:
import country_converter as coco
cc = coco.CountryConverter()
logging.getLogger('country_converter').setLevel(logging.ERROR)


import geonamescache
gc = geonamescache.GeonamesCache()
all_cities_raw = gc.get_cities()
city_lookup = defaultdict(list)
for city_id, info in all_cities_raw.items():
    city_lookup[info['name'].lower()].append(info)

In [3]:
from ensure_dataset import ensure_download
ensure_download()

dataset already exists: ../data/un-general-debates.csv


In [4]:
# load dataset
data = pd.read_csv("../data/un-general-debates.csv")
data.head()

,session,year,country,text
0,44,1989,MDV,﻿It is indeed a pleasure for me and the member...
1,44,1989,FIN,"﻿\nMay I begin by congratulating you. Sir, on ..."
2,44,1989,NER,"﻿\nMr. President, it is a particular pleasure ..."
3,44,1989,URY,﻿\nDuring the debate at the fortieth session o...
4,44,1989,ZWE,﻿I should like at the outset to express my del...


In [5]:
# load models

nlp = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

labels = nlp.get_pipe("ner").labels
print(labels)
print(f"GPE: {spacy.explain('GPE')}")

('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')
GPE: Countries, cities, states


In [6]:
# str.join(", ", data["country"].unique())

BLACKLIST = {
    "states", "member states", "the states", "the member states", 
    "hall", "apartheid", "patience", "tolerance", "ideas", "line"
}

def get_country_code(text):
    if isinstance(text, list):
        text = text[0] if text else None
    
    if not text or not isinstance(text, str): 
        return None
    
    clean_text = re.sub(r'^the\s+', '', text, flags=re.IGNORECASE).strip()
    clean_lower = clean_text.lower()
    
    if clean_lower in BLACKLIST:
        return None

    manual_map = {
        "soviet union": "SUN", "ussr": "SUN",
        "yugoslavia": "YUG", "german democratic republic": "DDR"
    }
    if clean_lower in manual_map:
        return manual_map[clean_lower]

    iso3 = cc.convert(names=clean_text, to='ISO3', not_found=None)
    
    if isinstance(iso3, list):
        iso3 = iso3[0]

    if iso3 and iso3 != "not found":
        return iso3

    return None

In [7]:
file_path = "../data/sentimented.json"

def is_country(name):
    code = get_country_code(name)
    return code is not None and len(cc.convert(names=code, to='ISO3')) == 3

def toSentiments(speech: str):
    if not isinstance(speech, str):
        return []
    
    doc = nlp(speech) 
    results = []
    for sent in doc.sents:
        countries = []
        for ent in sent.ents:
            if ent.label_ == "GPE":
                code = get_country_code(ent.text)
                if is_country(code):
                    countries.append(code)
                # print(f"Found: {countries}") 
                                
        if countries:
            score = analyzer.polarity_scores(sent.text)['compound']
            if score == 0:
                continue
            
            for country in countries:
                results.append((country, score))
    return results

if os.path.exists(file_path):
    with open(file_path, 'r') as file:
        print(f"sentimented data already exists: '{file_path}'. Loading instead of processing again")
        sentiments = json.load(file)
else:
    # data = data.head(50).copy()
    
    # raw_texts = data["text"][7000:].tolist()
    raw_texts = data["text"].tolist()
    with Pool() as pool: # Parallel
        nested_results = list(tqdm(pool.imap(toSentiments, raw_texts), total=len(raw_texts)))
        # print(nested_results)
    
    # restructure to {country: [country, scores]}
    sentiments_dict = defaultdict(list)
    for i, speech_results in enumerate(tqdm(nested_results)):
        speaker = data["country"][i]
        if not is_country(speaker):
            continue
        
        for country, score in speech_results:
            sentiments_dict[speaker].append([get_country_code(country), score])
    sentiments = dict(sentiments_dict)
        
    with open(file_path, 'w') as file:
        json.dump(sentiments, file, indent=1)
        
print(str(sentiments)[:100] + "...")

100%|██████████| 7507/7507 [02:28<00:00, 50.48it/s]


{'MDV': [['NAM', 0.8201], ['NAM', 0.4019], ['NAM', 0.4019], ['ZAF', 0.4588], ['ZAF', -0.4939], ['PSE...


In [8]:
# # Remove non GPE sentences
# print(f"before purge: {len(data)}")
# data = data[data['sentiments'] != "[]"]
# print(f"after purge: {len(data)}")
# data.head()

In [9]:
# Combine everything!!!
# data['sentiments'][0][0][0]
# [{country: [country, scores]}]

sentiTotal = []
for fromCountry, toCountriesSentiments in tqdm(sentiments.items()):
    sentiSum = {}
    # print(toCountriesSentiments)
    for speechSentiment in toCountriesSentiments:
        # print(speechSentiment)
        label = speechSentiment[0]
        score = speechSentiment[1]
        # print(label)
        
        sentiSum[label] = sentiSum.get(label, 0) + score
    sentiTotal.append({fromCountry: sentiSum})
    
    # if len(sentiTotal) > 20:
    #     break # debugging

print(sentiTotal)
sorted_data = sorted(sentiTotal, key=lambda x: next(iter(x)))
for x in sorted_data:
    print(x)

100%|██████████| 194/194 [00:00<00:00, 4249.81it/s]

[{'MDV': {'NAM': 8.014700000000001, 'ZAF': -3.1087000000000007, 'PSE': 10.4002, 'ISR': -7.0354, 'LBN': -6.6002, 'IRN': -1.4436, 'IRQ': -1.5495999999999999, 'AFG': 4.783399999999999, 'SYR': -1.1271, 'USA': 13.556000000000001, 'QAT': 0.8126, 'IND': -0.43370000000000003, 'PAK': 0.11360000000000003, 'SAU': -0.2644000000000001, 'MHL': 0.802, 'KOR': 9.7012, 'EST': 0.7579, 'LVA': 0.7579, 'LTU': 0.7579, 'KWT': 2.4134, 'BRA': 0.4215, 'BGD': 0.1351, 'ISL': 0.7351, 'CZE': 2.3672, 'CHE': 0.8258, 'MYS': 0.4404, 'PRT': 0.6808, 'ITA': 0.4588, 'BIH': 0.4216999999999998, 'TUN': 1.7875999999999999, 'EGY': 2.5227, 'LBY': 0.8015, 'MDV': 2.3038, 'COL': 1.9108, 'SLB': 0.6705, 'ZWE': 3.0383999999999998, 'AGO': -1.5299, 'IDN': -0.1069, 'NCL': 1.2145, 'JPN': 1.0670000000000002, 'LKA': 0.7906, 'KIR': 1.525, 'JOR': 0.7351, 'XKX': 3.5468, 'ZMB': 0.8126, 'PAN': 0.4767, 'CYP': -0.11199999999999999, 'MMR': 1.1875, 'SYC': 0.6597, 'AUS': 0.862, 'KHM': -0.18109999999999998, 'TZA': 1.7847, 'NRU': 0.8442, 'TON': 0.8442, 